# 04 — Train everything in one session (condensed: both detector arms + confidence head)

The merged, condensed replacement for running `01` twice plus `02` — built
because Kaggle allows **one active GPU session at a time**, so splitting the
GPU work across notebooks buys no parallelism and pays the ~20-minute setup
(detectron2 build + weight download) once per notebook instead of once total.

**The condensed recipe, and why it is defensible rather than just cheaper:**

1. **10,000 iterations instead of 40,000, with a real LR schedule.** The
   upstream config's `MAX_ITER: 40000` inherits `STEPS: (210000, 250000)` from
   its COCO base config — decay milestones *past the end of training*, so the
   40k run never anneals at all. It is a chopped COCO config, not a tuned
   recipe. 10k at batch 1 is ~18 epochs of fine-tuning from ImageNet-22k
   weights on ~564 images, with decay at 75%/90% — a normal fine-tuning budget.
2. **Swin stages 0–2 frozen; last stage + FPN + detection head train.** The
   backbone is the pretrained part (ImageNet-22k) and most of the compute;
   freezing its first three stages skips backward through the 18-block third
   stage, cuts activation memory, and shrinks checkpoints (AdamW keeps state
   only for trainable params). Set `FREEZE_STAGES = 0` to disable.
3. **Both arms use the identical recipe** — the robustness claim is the
   *difference* between the arms, and that comparison is only valid if the
   only difference is `degrade_prob`.
4. **The confidence head trains against the real trained backbone's FPN p5
   features** — an upgrade over notebook 02, which used a `TinyTrunk` stand-in
   precisely because no trained backbone existed yet. One head per arm is
   saved; notebook 05's `DetectorChannel` needs the arm-matched pair.

**Budget** (from the one real measurement: 2.7 s/iter unfrozen on a Kaggle T4;
frozen should be faster — the probe below measures it): setup ~25 min, ~4–5 h
per arm, head ~40 min → **~10–11 h, one 12 h session**. If the probe projects
past ~10.5 h, it prints a smaller `MAX_ITER` to use.

**Disk**: the 20 GB `/kaggle/working` cap killed a real run of `01` (see
`kaggle/README.md`). Here: `max_to_keep=1` during training, and each arm keeps
only `model_final.pth` when it finishes.

**If the session dies partway**: re-run top to bottom. A finished arm
(`model_final.pth` present) is skipped; a half-finished arm resumes from its
last checkpoint (`resume_or_load(resume=True)`, tested in `01`). Across
sessions, Save Version first, attach the old version's output, and copy each
arm's newest checkpoint into `/kaggle/working/checkpoints_<arm>/`.

**Honest verification status**: unlike notebooks 00–03, this notebook has NOT
been executed end-to-end (it needs a GPU session by definition). The dataset
mapper, registration, weight conversion and confidence-head training loop are
verbatim from 00–03, which were run-verified; the *new* glue — the freeze
call, the real-feature extraction, the per-arm loop — is verified against the
vendored source by reading (`hierarchialdet/swintransformer.py:_freeze_stages`)
and by the repo's AST preflight (`pytest tests/test_kaggle_notebooks.py`), but
first executes on Kaggle. Run notebook 00 first on a fresh setup; if something
breaks cheaply, it breaks there.

## 1. Setup — repo, detectron2, HierarchicalDet (condensed from 00; see 00 if anything fails)

In [ ]:
# Idempotent bootstrap -- safe to re-run (session restart, or you ran the cell twice).
# The old version was `!git clone` + `%cd`: it errored on the second run, and then
# left the notebook in the wrong directory with every relative path quietly broken.
import os, subprocess, sys

REPO_URL = "https://github.com/AIscend-Research/dental-extension.git"
if not os.path.exists("src/data/degradation.py"):          # not already at the repo root
    if not os.path.isdir("dental-extension"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("dental-extension")
sys.path.insert(0, ".")

from src.utils.kaggle_env import install_deps, summarize_environment

# Installs ONLY what's missing, with numpy/torch pinned to the image's versions.
# Do NOT `pip install -r requirements-core.txt` here: it can upgrade numpy, and
# Kaggle's torch -- plus the detectron2 you are about to build against it -- is
# compiled for the numpy already in the image. The upgrade "succeeds", then torch
# dies at import with "compiled using NumPy 1.x cannot be run in NumPy 2.x".
print("installed:", install_deps() or "nothing needed -- image already has it")
for k, v in summarize_environment().items():
    print(f"  {k}: {v}")

# Build detectron2 against the image's torch. Never reinstall torch on Kaggle.
# Deliberately NOT -q: this compiles for ~10 minutes, and a silent cell that long
# is indistinguishable from a hang.
import torch
print(f"building detectron2 against torch {torch.__version__} -- expect ~10 min")

!pip install -q ninja
!pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

!bash scripts/clone_baseline.sh  # import_hierarchicaldet needs external/HierarchicalDet to exist first

from src.utils.kaggle_env import import_hierarchicaldet

# Imports the real detectron2/pycocotools BEFORE putting HierarchicalDet on
# sys.path, so its vendored (uncompiled) copies cannot shadow them.
add_diffusiondet_config = import_hierarchicaldet()
print("hierarchialdet imports OK, using the real detectron2")

In [ ]:
import os
os.makedirs("models_weights", exist_ok=True)
if not os.path.exists("models_weights/swin_large_patch4_window7_224_22k.pkl"):
    # --fail: without it curl writes the error page to disk on a failed
    # download, and torch.load then dies with an unrelated-looking error
    # instead of "the download failed". Matches notebook 00.
    !curl -sL --fail "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window7_224_22k.pth" \
      -o models_weights/swin_large_patch4_window7_224_22k_raw.pth
    import pickle
    RAW = "models_weights/swin_large_patch4_window7_224_22k_raw.pth"
    assert os.path.exists(RAW) and os.path.getsize(RAW) > 1e8, \
        "backbone download failed or was truncated -- is notebook internet enabled?"
    ckpt = torch.load(RAW, map_location="cpu", weights_only=False)
    assert ckpt["model"]["patch_embed.proj.weight"].shape[0] == 192, "expected Swin-Large's 192-dim embedding"
    converted = {"model": ckpt["model"], "__author__": "third_party", "matching_heuristics": True}
    with open("models_weights/swin_large_patch4_window7_224_22k.pkl", "wb") as f:
        pickle.dump(converted, f)
    os.remove(RAW)
print("backbone weights ready")

## 2. Dataset

In [ ]:
from src.utils.kaggle_env import find_dentex_root

# Finds DENTEX wherever it is mounted rather than assuming a dataset slug -- the
# path depends on what you named the Kaggle Dataset. If it is not attached, the
# error tells you how to attach it.
DATA_ROOT = str(find_dentex_root())
print("DATA_ROOT:", DATA_ROOT)

## 3. Register the dataset and define the custom mapper

Verbatim from notebook 01 (which found and fixed the upstream mapper's
hardcoded-paths crash and the silent box/label misalignment — see
`kaggle/README.md`). `degrade_prob` is what separates the two arms.

In [ ]:
import sys, warnings, copy
warnings.filterwarnings("ignore")

import cv2
import numpy as np
from detectron2.structures import Instances, Boxes
from detectron2.data import DatasetCatalog

sys.path.insert(0, ".")
from src.data.degradation import apply_degradations
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
print("registered:", {k: len(v) for k, v in split.items()})


class CariesDatasetMapper:
    """Reads DENTEX's hierarchical annotations into DiffusionDet's expected
    Instances fields. Ground-truth boxes only -- no pretrained-box curriculum
    (see kaggle/01's notes for why that's the right call here, not a corner cut).

    degrade_prob controls the robustness arm: 0.0 trains on clean images (the
    baseline), >0.0 applies src/data/degradation.py to that fraction of
    training images, with ground-truth boxes remapped through the same
    geometric warp. Keeping some clean images in the mix (prob < 1.0) is
    deliberate -- the deployed model still sees good photos, and an all-
    degraded diet would trade clean-image accuracy away for nothing.
    """

    def __init__(self, target_size=800, is_train=True, degrade_prob=0.0,
                 severity_range=(0.3, 0.9), max_simultaneous=3):
        self.target_size = target_size
        self.is_train = is_train
        self.degrade_prob = degrade_prob
        self.severity_range = severity_range
        self.max_simultaneous = max_simultaneous

    def __call__(self, d):
        d = copy.deepcopy(d)
        img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
        h0, w0 = img.shape[:2]

        annos = [a for a in d.get("annotations", []) if not a.get("iscrowd", 0)]
        boxes_xywh = np.array([a["bbox"] for a in annos], dtype=np.float64).reshape(-1, 4)

        # --- robustness arm: degrade the image AND remap the boxes with it ---
        if self.is_train and self.degrade_prob > 0 and np.random.rand() < self.degrade_prob:
            res = apply_degradations(
                img,
                severity_range=self.severity_range,
                max_simultaneous=self.max_simultaneous,
                boxes=boxes_xywh,
            )
            img = res.image
            boxes_xywh = res.boxes
            # a box warped out of frame comes back zero-area -- drop it and its
            # labels together, or the class lists desynchronize from the boxes
            keep = (boxes_xywh[:, 2] > 1.0) & (boxes_xywh[:, 3] > 1.0)
            boxes_xywh = boxes_xywh[keep]
            annos = [a for a, k in zip(annos, keep) if k]

        img = cv2.resize(img, (self.target_size, self.target_size))
        scale_x, scale_y = self.target_size / w0, self.target_size / h0

        out = {
            "image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)),
            "height": self.target_size, "width": self.target_size,
        }
        if not self.is_train:
            return out

        inst = Instances((self.target_size, self.target_size))
        boxes, c1, c2, c3 = [], [], [], []
        for ann, (x, y, w, h) in zip(annos, boxes_xywh):
            boxes.append([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y])
            c1.append(ann["category_id_1"]); c2.append(ann["category_id_2"]); c3.append(ann["category_id_3"])
        inst.gt_boxes = Boxes(torch.tensor(boxes, dtype=torch.float32)) if boxes else Boxes(torch.zeros(0, 4))
        inst.gt_classes_1 = torch.tensor(c1, dtype=torch.int64)
        inst.gt_classes_2 = torch.tensor(c2, dtype=torch.int64)
        inst.gt_classes_3 = torch.tensor(c3, dtype=torch.int64)
        out["instances"] = inst
        return out


# Sanity check the degraded path before spending GPU hours on it: boxes must
# move with the image, and must stay inside the frame.
_probe = CariesDatasetMapper(is_train=True, degrade_prob=1.0)(DatasetCatalog.get("custom_train_class")[0])
_pb = _probe["instances"].gt_boxes.tensor
print("degraded-sample check -- boxes:", tuple(_pb.shape),
      "| in frame:", bool(((_pb >= 0) & (_pb <= 800)).all()))

## 4. Config and trainer — the condensed recipe

Same `CariesTrainer` as 01 plus two changes: the checkpointer keeps only the
newest checkpoint (the 20 GB disk cap is a confirmed session-killer), and
`build_model` freezes Swin stages 0–2 via the vendored backbone's own
`_freeze_stages()` before the optimizer is built (detectron2's optimizer
builder skips `requires_grad=False` params, so frozen weights also drop out
of the AdamW state and the checkpoints shrink).

The vendored freeze convention (`hierarchialdet/swintransformer.py`):
`frozen_stages = N` freezes `patch_embed` plus `layers[0 .. N-2]`. So
"freeze stages 0–2, train stage 3" = `frozen_stages = 4` = `FREEZE_STAGES + 1`
with our `FREEZE_STAGES = 3`. The cell prints the frozen/total param count —
**check it says roughly 100M of 282M trainable**; ~0M or ~282M means the
mapping drifted and the run would silently be a different experiment.

In [ ]:
import logging
# detectron2 logs the ENTIRE model architecture on every trainer construction;
# silence its logger (not the root logger) -- real errors still raise.
logging.getLogger("detectron2").setLevel(logging.WARNING)
logging.getLogger("fvcore").setLevel(logging.WARNING)

from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader

MAX_ITER = 10000        # ~18 epochs at batch 1 -- see the header for why not 40k
FREEZE_STAGES = 3       # Swin stages 0-2 frozen; 0 = train everything (01's behavior)
CHECKPOINT_PERIOD = 500
ARMS_TO_TRAIN = ["baseline", "robustness"]
DEGRADE_PROB = {"baseline": 0.0, "robustness": 0.7}

# which arm the train loader builds for -- set by the probe and the arm loop below
CURRENT_ARM = ARMS_TO_TRAIN[0]


def freeze_swin_stages(model, n_stages):
    """Freeze patch_embed + Swin stages [0..n_stages-1] via the vendored
    backbone's own _freeze_stages(). Their convention: frozen_stages=N freezes
    patch_embed plus layers[0..N-2], hence the +1."""
    swin = model.backbone.bottom_up
    swin.frozen_stages = n_stages + 1
    swin._freeze_stages()
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"froze {frozen/1e6:.1f}M of {total/1e6:.1f}M params "
          f"({(total-frozen)/1e6:.1f}M trainable) -- expect roughly 100M trainable; "
          "~0M or ~282M means the freeze mapping drifted, stop and check")
    return model


class CariesTrainer(DefaultTrainer):
    @classmethod
    def build_model(cls, cfg):
        # freeze BEFORE DefaultTrainer builds the optimizer, so frozen params
        # are excluded from it (smaller checkpoints, no wasted state)
        model = super().build_model(cfg)
        if FREEZE_STAGES > 0:
            freeze_swin_stages(model, FREEZE_STAGES)
        return model

    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(
            cfg, mapper=CariesDatasetMapper(is_train=True, degrade_prob=DEGRADE_PROB[CURRENT_ARM])
        )

    def build_hooks(self):
        from detectron2.engine import hooks as d2_hooks
        # metrics every 100 iters instead of 20 -- 20 floods the output pane
        ret = [h for h in super().build_hooks() if not isinstance(h, d2_hooks.PeriodicWriter)]
        ret.append(d2_hooks.PeriodicWriter(self.build_writers(), period=100))
        # keep ONLY the newest checkpoint: unbounded retention filled the 20 GB
        # working-dir cap and killed a real run (kaggle/README.md). Resume only
        # ever needs the newest one; model_final.pth is never purged.
        for h in ret:
            if isinstance(h, d2_hooks.PeriodicCheckpointer):
                h.max_to_keep = 1
                h.recent_checkpoints = []
        return ret


def make_cfg(arm):
    cfg = get_cfg()
    add_diffusiondet_config(cfg)
    cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
    cfg.MODEL.WEIGHTS = "models_weights/swin_large_patch4_window7_224_22k.pkl"
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.DATASETS.TRAIN = ("custom_train_class",)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 0  # >0 crashed in dev (macOS spawn); untested on Kaggle -- see kaggle/README.md
    cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"  # config default "full_model" is invalid in this detectron2
    cfg.SOLVER.IMS_PER_BATCH = 1  # 2 OOM'd UNFROZEN on a T4; freezing frees activation memory,
    # so 2 may now fit -- if you try it, halve MAX_ITER to keep the epoch count comparable
    cfg.SOLVER.AMP.ENABLED = True
    cfg.SOLVER.MAX_ITER = MAX_ITER
    # the config inherits STEPS=(210000, 250000) -- decay milestones past the end
    # of training, i.e. no annealing at all. Scale them to the actual run length.
    cfg.SOLVER.STEPS = (int(MAX_ITER * 0.75), int(MAX_ITER * 0.90))
    cfg.SOLVER.CHECKPOINT_PERIOD = CHECKPOINT_PERIOD
    # per-arm output dir so the arms cannot overwrite each other's checkpoints
    cfg.OUTPUT_DIR = f"/kaggle/working/checkpoints_{arm}"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg

print(f"recipe: MAX_ITER={MAX_ITER}, STEPS={(int(MAX_ITER*0.75), int(MAX_ITER*0.9))}, "
      f"FREEZE_STAGES={FREEZE_STAGES}, arms={ARMS_TO_TRAIN}")

## 5. Throughput probe — read the number before walking away

Times a few real steps of the frozen-backbone recipe in a scratch dir (so it
can't leave a `model_final.pth` that the arm loop below would mistake for a
finished arm). The only prior measurement is 2.7 s/iter **unfrozen**; the
frozen number is what your session budget actually depends on.

In [ ]:
import shutil, time

PROBE_ITERS = 15
CURRENT_ARM = "baseline"

cfg = make_cfg("baseline")
cfg.OUTPUT_DIR = "/kaggle/working/probe_scratch"   # never mixes with a real arm's dir
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
cfg.SOLVER.MAX_ITER = PROBE_ITERS

trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=False)
t0 = time.time()
trainer.train()
dt = time.time() - t0
del trainer
torch.cuda.empty_cache()
shutil.rmtree(cfg.OUTPUT_DIR, ignore_errors=True)

per_iter = dt / PROBE_ITERS  # includes warmup overhead -> a slightly pessimistic estimate
per_arm_h = MAX_ITER * per_iter / 3600
total_h = 2 * per_arm_h + 1.2   # + head training + margin
print(f"measured: {per_iter:.2f}s/iteration (frozen recipe, this GPU)")
print(f"projected: {per_arm_h:.1f}h per arm -> both arms + confidence head ~= {total_h:.1f}h")
if total_h > 10.5:
    fit_iters = int((10.5 - 1.2) / 2 * 3600 / per_iter)
    print(f"WARNING: does not fit a 12h session with margin. Set MAX_ITER = {fit_iters} "
          "in section 4 (STEPS rescale automatically) and rerun from there.")

## 6. Train both arms

Sequential, one shared setup. Idempotent: a finished arm (its
`model_final.pth` exists) is skipped, a half-finished arm resumes from its
newest checkpoint. After each arm, periodic checkpoints are deleted —
notebook 05 reads only `model_final.pth`, and disk is the known killer.

In [ ]:
import glob

for arm in ARMS_TO_TRAIN:
    out_dir = f"/kaggle/working/checkpoints_{arm}"
    final = os.path.join(out_dir, "model_final.pth")
    if os.path.exists(final):
        print(f"[{arm}] model_final.pth already exists -- skipping (delete it to retrain)")
        continue

    CURRENT_ARM = arm
    cfg = make_cfg(arm)
    print(f"\n=== arm: {arm} (degrade_prob={DEGRADE_PROB[arm]}) -> {cfg.OUTPUT_DIR} ===")
    trainer = CariesTrainer(cfg)
    trainer.resume_or_load(resume=True)  # continues a half-finished arm; fresh start otherwise
    trainer.train()
    print(f"[{arm}] done at iteration {trainer.iter}")
    del trainer
    torch.cuda.empty_cache()

    # keep only model_final.pth -- periodic checkpoints are ~2 GB each and 05 never reads them
    for f in glob.glob(os.path.join(out_dir, "model_0*.pth")):
        os.remove(f)
    print(f"[{arm}] kept:", sorted(os.listdir(out_dir)))

## 7. Confidence head — against the REAL trained backbone's features

The upgrade notebook 02 explicitly deferred: 02 trained against a `TinyTrunk`
stand-in because no trained backbone existed. Both exist now, so this trains
one head per arm on that arm's frozen FPN p5 features (256-dim, exactly what
`ConfidenceHead(in_features=256)` was designed for, and what
`DetectorChannel.read()` will feed it in notebook 05).

Same data recipe and best-by-val-loss selection as 02 (val loss is genuinely
unstable on this task — reporting the last epoch makes the numbers a coin
flip; these are model-selected on val, say so when quoting them). Compare
against 02's stand-in-trunk numbers in
`docs/phase3_confidence_head_training.md`: ~77% dominant-degradation accuracy
at 495 images, usability correlation 0.93.

In [ ]:
import random
import torch.nn as nn
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from src.models.confidence_head import ConfidenceHead
from src.utils.seed import set_seed

IMG_SIZE = 800          # the detector's input size -- features must come from the geometry it was trained at
VARIANTS_PER_IMAGE = 4
N_TRAIN_IMAGES = 500
N_VAL_IMAGES = 100
FEAT_BATCH = 4
N_EPOCHS = 15
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_ROOT = f"{DATA_ROOT}/xrays"
id_to_name = {im["id"]: im["file_name"] for im in coco["images"]}
train_files = [id_to_name[i] for i in split["train"][:N_TRAIN_IMAGES]]
val_files = [id_to_name[i] for i in split["val"][:N_VAL_IMAGES]]


def build_examples(file_names, seed):
    rng = random.Random(seed)
    examples = []
    for fn in file_names:
        base = cv2.resize(cv2.imread(os.path.join(IMAGE_ROOT, fn), cv2.IMREAD_COLOR),
                          (IMG_SIZE, IMG_SIZE))
        examples.append((base, np.zeros(len(_DEG_NAMES), dtype=np.float32)))
        for _ in range(VARIANTS_PER_IMAGE - 1):
            result = apply_degradations(base, seed=rng.randint(0, 1_000_000))
            examples.append((result.image, result.label_vector()))
    return examples


from src.data.degradation import DEGRADATION_NAMES as _DEG_NAMES

set_seed(0)
train_examples = build_examples(train_files, seed=1)
val_examples = build_examples(val_files, seed=2)
train_y = torch.from_numpy(np.stack([e[1] for e in train_examples]))
val_y = torch.from_numpy(np.stack([e[1] for e in val_examples]))
train_usability = 1.0 - train_y.max(dim=1).values
val_usability = 1.0 - val_y.max(dim=1).values
print(f"examples: {len(train_examples)} train / {len(val_examples)} val")


def extract_p5(model, cfg_arm, examples):
    """Frozen FPN p5 features for each example image; cached on CPU."""
    pixel_mean = torch.tensor(cfg_arm.MODEL.PIXEL_MEAN).view(1, 3, 1, 1).to(DEVICE)
    pixel_std = torch.tensor(cfg_arm.MODEL.PIXEL_STD).view(1, 3, 1, 1).to(DEVICE)
    feats = []
    with torch.no_grad():
        for i in range(0, len(examples), FEAT_BATCH):
            batch = np.stack([e[0] for e in examples[i:i + FEAT_BATCH]]).astype(np.float32)
            x = torch.from_numpy(batch).permute(0, 3, 1, 2).to(DEVICE)
            x = (x - pixel_mean) / pixel_std
            feats.append(model.backbone(x)["p5"].cpu())
    return torch.cat(feats)


loss_fn = nn.SmoothL1Loss()
for arm in ARMS_TO_TRAIN:
    final = f"/kaggle/working/checkpoints_{arm}/model_final.pth"
    if not os.path.exists(final):
        print(f"[{arm}] no model_final.pth -- arm not trained, skipping its head")
        continue

    print(f"\n=== confidence head for arm: {arm} ===")
    cfg_arm = make_cfg(arm)
    cfg_arm.MODEL.WEIGHTS = final
    det = build_model(cfg_arm)
    DetectionCheckpointer(det).load(final)
    det.eval()
    train_feats = extract_p5(det, cfg_arm, train_examples)
    val_feats = extract_p5(det, cfg_arm, val_examples)
    del det
    torch.cuda.empty_cache()
    print(f"  features: train {tuple(train_feats.shape)}, val {tuple(val_feats.shape)}")

    head = ConfidenceHead(in_features=256).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
    n_train = train_feats.shape[0]
    best_val_loss, best_epoch, best_state = float("inf"), -1, None
    vy, vu = val_y.to(DEVICE), val_usability.to(DEVICE)

    for epoch in range(N_EPOCHS):
        head.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0.0
        for i in range(0, n_train, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            optimizer.zero_grad()
            severity_pred, usability_pred = head(train_feats[idx].to(DEVICE))
            loss = (loss_fn(severity_pred, train_y[idx].to(DEVICE))
                    + loss_fn(usability_pred, train_usability[idx].to(DEVICE)))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * idx.shape[0]
        epoch_loss /= n_train

        head.eval()
        with torch.no_grad():
            vs, vu_pred = head(val_feats.to(DEVICE))
            val_loss = (loss_fn(vs, vy) + loss_fn(vu_pred, vu)).item()
        marker = ""
        if val_loss < best_val_loss:
            best_val_loss, best_epoch = val_loss, epoch + 1
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            marker = "  <- best so far"
        print(f"  epoch {epoch+1:2d}/{N_EPOCHS}  train={epoch_loss:.4f}  val={val_loss:.4f}{marker}")

    head.load_state_dict(best_state)
    head.eval()
    with torch.no_grad():
        vs, vu_pred = head(val_feats.to(DEVICE))
    nonclean = vy.max(dim=1).values > 0
    dom_acc = (vy[nonclean].argmax(dim=1) == vs[nonclean].argmax(dim=1)).float().mean().item()
    corr = np.corrcoef(vu_pred.cpu().numpy(), vu.cpu().numpy())[0, 1]
    print(f"  [best epoch {best_epoch}] dominant-degradation acc: {dom_acc:.3f} "
          f"(chance {1/len(_DEG_NAMES):.2f}) | usability corr: {corr:.3f}")
    out_path = f"/kaggle/working/confidence_head_{arm}.pth"
    torch.save(best_state, out_path)
    print(f"  saved -> {out_path}")

## 8. Done — Save Version, then notebook 05

**Save Version now** (Quick Save is fine) — `/kaggle/working/` is only
preserved as attachable output if you do. This version's output should hold:

- `checkpoints_baseline/model_final.pth` and `checkpoints_robustness/model_final.pth`
- `confidence_head_baseline.pth` and `confidence_head_robustness.pth`

Then open `05_evaluate_all.ipynb`, attach this notebook's output as a data
source (Add Data → Your Work → Notebook Output Files) plus the DENTEX
dataset, and run it — it finds these files by glob, no slug to configure.